# SeguroData Bogotá
## Plan de Proyecto + Catálogo de Fuentes de Datos
### Concurso Datos al Ecosistema 2026 · MinTIC · Reto #2 · Seguridad Ciudadana · Nivel Medio

---

| Campo | Detalle |
|-------|---------|
| **Proyecto** | SeguroData Bogotá |
| **Concurso** | Datos al Ecosistema 2026 — MinTIC |
| **Reto** | #2 — Seguridad Ciudadana y Justicia |
| **Nivel** | Medio |
| **Ciudad piloto** | Bogotá D.C. |
| **Unidad de análisis** | UPZ (112 zonas de Bogotá) |
| **Tecnologías** | Python · XGBoost · Claude API · Streamlit · GeoPandas |
| **Entrega GitHub** | 13 de julio de 2026 |

---

> **Este notebook** es el documento maestro del proyecto. Contiene:  
> 1. Descripción del problema y la propuesta  
> 2. Usuarios del sistema  
> 3. Los 4 módulos que se construirán  
> 4. **Catálogo detallado de las 12 fuentes de datos** (F1-F10 activas + F11-F12 planificadas) — URLs, variables, frecuencia, advertencias  
> 5. Arquitectura técnica  
> 6. Cronograma de fases  
> 7. Criterios de evaluación del concurso

---
## 📌 SECCIÓN 1 — El Problema

### ¿Por qué este proyecto?

Bogotá registra cientos de miles de hurtos y delitos al año, pero las decisiones de patrullaje todavía se toman con información atrasada o basada en la experiencia del comandante, no en datos.

**El problema tiene tres caras:**

- **Reactivo:** la Policía actúa cuando el delito ya ocurrió, no antes.
- **Sin granularidad:** los reportes llegan a nivel ciudad o localidad — no a nivel de barrio y hora.
- **Sin conexión:** los datos existen (están en datos.gov.co), pero nadie los ha integrado en una herramienta operativa lista para usar en un CAI.

### ¿Qué resuelve SeguroData?

Un sistema que responde **tres preguntas concretas**, en ese orden:

| Pregunta | Módulo | Respuesta |
|----------|--------|-----------|
| ¿Qué está pasando? | Diagnóstico | Mapa de hurtos por UPZ, horario y tipo |
| ¿Qué va a pasar? | Predicción | Riesgo ALTO / MEDIO / BAJO por UPZ en las próximas 48h |
| ¿Qué hacer? | Recomendación | IA explica la causa y dice en qué CAI reforzar |


---
## 👤 SECCIÓN 2 — Usuarios del Sistema

El sistema tiene **tres perfiles de usuario** bien definidos, cada uno con una vista distinta en el dashboard.

---

### 🔴 Usuario Principal — Comandante de Cuadrante / CAI

**Quién es:** Policía Metropolitana de Bogotá (MEBOG). El oficial que decide dónde desplegar pie de fuerza antes de cada turno.

**Qué necesita del sistema:**
- Mapa de riesgo de su zona para las próximas 6–12 horas
- Una recomendación clara: *"Refuerce el cuadrante 07 esta noche — alta concentración de hurtos prevista en Chapinero"*
- Número y dirección del CAI asignado a cada UPZ

**Por qué es el usuario más importante:** Si este perfil encuentra el sistema útil, el impacto es inmediato y medible. El jurado valora exactamente esto.

---

### 🔵 Usuario Estratégico — Secretaría Distrital de Seguridad

**Quién es:** Funcionario público que formula política de seguridad para Bogotá. Toma decisiones de mediano plazo.

**Qué necesita del sistema:**
- Dashboard ejecutivo con KPIs por localidad
- Tendencia histórica 2020–2024 y variación anual
- Reporte automático en lenguaje natural generado por IA

---

### 🟢 Usuario Ciudadano — Habitante de Bogotá

**Quién es:** Cualquier persona que quiera saber si su barrio es seguro antes de salir.

**Qué necesita del sistema:**
- *"¿Qué tan seguro está Kennedy hoy?"* → respuesta simple
- Horarios de mayor riesgo en su UPZ
- Número del CAI más cercano


---
## 🧩 SECCIÓN 3 — Los 4 Módulos del Sistema

Todo el sistema vive en **una sola aplicación Streamlit** publicada en la nube. Cuatro módulos, un solo dashboard.

---

### Módulo 1 · Diagnóstico — ¿Qué está pasando en Bogotá?

**Tipo:** Análisis descriptivo  
**Usuario principal:** Secretaría de Seguridad, ciudadano

Responde con datos y visualizaciones a las preguntas más básicas sobre la criminalidad en Bogotá:

- **Mapa de calor interactivo** de hurtos por UPZ — el usuario puede filtrar por año, tipo de delito y localidad
- **Top 5 localidades** con más delitos — gráfica de barras comparativa
- **Heatmap día × hora** — muestra en qué combinación día-hora ocurren más delitos
- **Tendencia anual 2020–2024** — con variación porcentual año a año

**Entregable:** Página "Diagnóstico" en el dashboard Streamlit con filtros interactivos.

---

### Módulo 2 · Predicción — ¿Dónde habrá riesgo mañana?

**Tipo:** Machine Learning  
**Algoritmo:** XGBoost (clasificación multiclase)  
**Usuario principal:** Comandante de cuadrante / CAI

El modelo clasifica cada UPZ como **ALTO / MEDIO / BAJO** riesgo para las próximas 48 horas, usando:

- Datos históricos de delitos en esa UPZ (últimas 4 semanas)
- Día de la semana y franja horaria
- Temperatura y precipitación (Open-Meteo, tiempo real)
- Estrato socioeconómico promedio de la UPZ

**Validación:** Backtesting con datos de 2024 — el modelo se entrena con 2020–2023 y se prueba contra lo que realmente pasó en 2024.

**Entregable:** Mapa predictivo con colores rojo/amarillo/verde por UPZ + tabla de UPZs en riesgo ALTO.

---

### Módulo 3 · Recomendación — ¿Qué hacer y en qué CAI?

**Tipo:** IA Generativa  
**Tecnología:** Claude API (Anthropic)  
**Usuario principal:** Comandante de cuadrante / CAI

Para cada UPZ clasificada como riesgo ALTO, la IA generativa produce un mensaje en lenguaje natural:

> *"La UPZ Chapinero (localidad Chapinero) presenta riesgo ALTO para esta noche. Los datos muestran alta concentración histórica de hurtos entre las 9pm y 1am, especialmente los viernes. Se prevé lluvia débil que puede aumentar el riesgo en zonas cubiertas (estaciones TM, centros comerciales). Se recomienda reforzar el cuadrante CHAP-07 asignado al CAI Chapinero (Cra 13 #55-40, tel. 601-123-4567) entre las 8pm y 2am."*

El mensaje se genera automáticamente a partir de los valores SHAP del modelo (qué variable causó el riesgo) y los datos del cuadrante policial.

**Entregable:** Sección "Recomendación del día" en el dashboard, actualizada automáticamente.

---

### Módulo 4 · Chatbot — Pregunta en lenguaje natural

**Tipo:** NLP + IA Generativa  
**Tecnología:** Claude API + RAG sobre datos de la ciudad  
**Usuario principal:** Ciudadano, funcionario

Cualquier usuario puede escribir una pregunta y el sistema responde con datos:

- *"¿Qué tan seguro está Kennedy hoy?"* → Respuesta con el nivel de riesgo actual
- *"¿A qué horas hay más hurtos en Chapinero?"* → Gráfica automática del heatmap hora
- *"¿Cuál es el CAI más cercano a la UPZ Teusaquillo?"* → Dirección y teléfono

**Entregable:** Pestaña "Consulta ciudadana" en el dashboard Streamlit.


---
## 📦 SECCIÓN 4 — Catálogo de Fuentes de Datos

### Convenciones de este catálogo

| Símbolo | Significado |
|---------|-------------|
| 🟢 VERDE | URL verificada y activa. Datos frescos. Listo para usar. |
| 🟡 AMARILLO | Verificar fecha de actualización antes de descargar. |
| 🔴 ROJO | Dato desactualizado (stale). Usar con precaución o sustituir. |

**Nivel medio requiere entre 3 y 10 datasets.** Usamos **10 fuentes activas** (F1-F10) + 2 planificadas (F11/F12 en Fases 2-3): un conjunto diverso de fuentes estructuradas (F1-F8 → XGBoost) y no estructuradas (F9-F10 → GraphRAG + Claude API).

---

---
### 🟢 FUENTE 1 — Delito de Alto Impacto · Bogotá D.C.

**Organización:** Secretaría Distrital de Seguridad, Convivencia y Justicia  
**Portal:** datosabiertos.bogota.gov.co  
**Plataforma:** CKAN — descarga de archivo GeoJSON comprimido en ZIP

**🔗 Enlace de descarga:**
```
https://datosabiertos.bogota.gov.co/dataset/7b270013-42ca-436b-9c1e-3bcb7d280c6b/resource/aba0e25d-d407-45f4-9a98-327493b538bd/download/dai_geojson.zip
```

**📋 Qué contiene este dataset:**  
Totales de delitos de alto impacto por **localidad × año** (2018–2026). Cubre las 21 localidades de Bogotá con datos agregados anuales por tipo de delito.

> **⚠️ HALLAZGO FASE 1B (27 mayo 2026):** Al descargar F1 se encontró que el dataset tiene **21 filas** (una por localidad), no 500K+ registros individuales. La granularidad es **localidad × año**, no UPZ. F1 **no tiene coordenadas GPS ni desglose por UPZ**. Por esta razón, F1 se usa únicamente como referencia de EDA histórico (tendencia 2018–2026 por localidad), pero **no es la fuente base del modelo**.  
>  
> **La fuente base del modelo es F5 (NUSE 123 — 128,314 registros con granularidad UPZ × mes × tipo).**

**📊 Rol real en el pipeline:**

| Uso | Descripción |
|-----|-------------|
| EDA histórico | Tendencia anual 2018–2026 por localidad (V4 del EDA) |
| Benchmarking | Comparar con Policía Nacional (F6) a nivel localidad |
| ~~Variable objetivo Y~~ | ❌ No posible — no tiene desglose UPZ |

**📐 Volumen real (Bronze):** 21 filas × 21 columnas (totales anuales por localidad)  
**🔄 Frecuencia:** Semestral | **🏷️ Licencia:** CC BY-SA 4.0  

**💡 Por qué sigue siendo útil:**  
Permite mostrar la tendencia histórica 2018–2026 en el EDA (Visualización V4), contextualizando los datos NUSE 2025–2026 dentro de una serie temporal más larga. También sirve para el benchmarking con F6 (Policía Nacional).

In [ ]:
# ──────────────────────────────────────────────────────────────
# FUENTE 1 — Carga del GeoJSON de Delito de Alto Impacto
# ──────────────────────────────────────────────────────────────
# INSTRUCCIONES DE DESCARGA:
# 1. Entra al enlace de la Fuente 1
# 2. El archivo se descarga automáticamente como dai_geojson.zip
# 3. Guárdalo en la carpeta datos/raw/
# 4. Ejecuta este bloque

import zipfile
import geopandas as gpd
import pandas as pd

RUTA_ZIP_DAI = 'datos/raw/dai_geojson.zip'

def cargar_delito_alto_impacto(ruta_zip):
    try:
        with zipfile.ZipFile(ruta_zip) as z:
            archivos = z.namelist()
            geojson = [f for f in archivos if f.endswith('.geojson') or f.endswith('.json')][0]
            with z.open(geojson) as f:
                gdf = gpd.read_file(f)
        print(f'✅ Delito de Alto Impacto cargado: {len(gdf):,} registros')
        print(f'   Columnas disponibles: {gdf.columns.tolist()}')
        print(f'   Tipos de delito: {gdf["tipologia_delito"].nunique() if "tipologia_delito" in gdf.columns else "ver columnas"}')
        return gdf
    except FileNotFoundError:
        print(f'⚠️  Archivo no encontrado en: {ruta_zip}')
        print('   Descarga el ZIP desde el enlace de FUENTE 1 y guárdalo en datos/raw/')
        return None

# gdf_dai = cargar_delito_alto_impacto(RUTA_ZIP_DAI)
print('▶ Descomenta la línea anterior cuando tengas el archivo descargado.')


---
### 🟢 FUENTE 2 — UPZ Shapefile · IDECA ⭐ BASE ESPACIAL OBLIGATORIA

**Organización:** IDECA — Instituto Distrital de Estadística, Cartografía y Análisis  
**Portal:** datosabiertos.bogota.gov.co  
**Plataforma:** CKAN — descarga directa GeoJSON

**🔗 Enlace de descarga:**
```
https://datosabiertos.bogota.gov.co/dataset/808582fc-ffc8-4649-8428-7e1fd8d3820c/resource/a5c8c591-0708-420f-8eb7-9f3147e21c40/download/unidadplaneamientolocal.json
```

**📋 Qué encontrarás en ese enlace:**  
Los 112 polígonos que definen las Unidades de Planeamiento Zonal (UPZ) de Bogotá en formato GeoJSON. Esta es la **capa base de todo el proyecto**: sin estos polígonos no se puede hacer ningún spatial join, no existe el mapa del dashboard y no hay unidad de análisis. Cada UPZ es una fila en el dataset de entrenamiento del modelo.

**📊 Variables que usaremos para el proyecto:**

| Variable | Descripción | Cómo la usamos |
|----------|-------------|----------------|
| `codigo_upz` | Código numérico único de la UPZ | Llave de unión con todas las demás fuentes |
| `nombre` | Nombre de la UPZ (ej. "Chapinero") | Etiquetas en el dashboard y chatbot |
| `geometry` | Polígono del territorio | Spatial joins, mapas de calor, mapa predictivo |
| `area` | Área en m² | Calcular densidad de delitos por km² |
| `localidad` | Localidad a la que pertenece la UPZ | Agrupación geográfica para el módulo diagnóstico |

**🔄 Frecuencia de actualización:** Según necesidad (última actualización: febrero 2025)  
**📁 Formato:** GeoJSON  
**📐 Volumen:** 112 polígonos  
**🏷️ Licencia:** CC BY 4.0  

**⚠️ Advertencia de calidad:**  
Verificar que el GeoJSON descargado tenga exactamente **112 polígonos**. Algunos portales confunden UPZ con UPL (Unidades de Planeamiento Local), que son 117.

**💡 Por qué es indispensable:**  
Sin esta capa no hay modelo. Todo el análisis —desde el mapa de calor hasta la predicción— se hace a nivel UPZ. Esta fuente define la unidad geográfica del proyecto.


In [ ]:
# ──────────────────────────────────────────────────────────────
# FUENTE 2 — Carga del Shapefile de UPZ desde la URL directa
# Se puede cargar directo desde la URL sin descargar manualmente
# ──────────────────────────────────────────────────────────────
import geopandas as gpd

URL_UPZ = (
    'https://datosabiertos.bogota.gov.co/dataset/'
    '808582fc-ffc8-4649-8428-7e1fd8d3820c/resource/'
    'a5c8c591-0708-420f-8eb7-9f3147e21c40/download/unidadplaneamientolocal.json'
)

def cargar_upz(url=URL_UPZ):
    try:
        upz = gpd.read_file(url)
        upz = upz.to_crs('EPSG:4326')  # Asegurar coordenadas WGS84
        print(f'✅ UPZ cargadas: {len(upz)} polígonos')
        if len(upz) != 112:
            print(f'   ⚠️  Atención: se esperaban 112, se encontraron {len(upz)}')
        print(f'   Columnas: {upz.columns.tolist()}')
        return upz
    except Exception as e:
        print(f'❌ Error al cargar UPZ: {e}')
        print('   Descarga manual desde el enlace de FUENTE 2.')
        return None

# upz = cargar_upz()
print('▶ Descomenta la línea anterior para cargar las UPZ.')


---
### 🟢 FUENTE 3 — Open-Meteo · Clima Bogotá ⭐ ÚNICA EN TIEMPO REAL

**Organización:** Open-Meteo (servicio internacional libre, sin ánimo de lucro)  
**Portal:** open-meteo.com  
**Plataforma:** REST API gratuita — sin registro, sin clave API

**🔗 Enlace de ejemplo (histórico Bogotá 2024):**
```
https://archive-api.open-meteo.com/v1/archive?latitude=4.6097&longitude=-74.0817&start_date=2024-01-01&end_date=2024-12-31&hourly=temperature_2m,precipitation,windspeed_10m,relativehumidity_2m&timezone=America/Bogota
```

**📋 Qué encontrarás en ese enlace:**  
Datos climáticos horarios de Bogotá desde 1940 hasta hoy. La respuesta llega en JSON y contiene una columna por variable climática, con un valor por hora. Para predicción en tiempo real se usa el endpoint `/forecast` (pronóstico de los próximos 7 días). Es la única fuente del proyecto que permite predicciones hora a hora con contexto climático real y actualizado.

**📊 Variables que usaremos para el proyecto:**

| Variable de la API | Descripción | Cómo la usamos |
|--------------------|-------------|----------------|
| `temperature_2m` | Temperatura en °C a 2 metros | Feature del modelo: temperatura alta → más riñas y lesiones |
| `precipitation` | Precipitación en mm por hora | Feature del modelo: lluvia → menor hurto callejero (menos peatones) |
| `windspeed_10m` | Velocidad del viento en km/h | Feature secundaria de contexto |
| `relativehumidity_2m` | Humedad relativa en % | Variable de contexto climático |
| `time` | Marca de tiempo ISO 8601 | Unión con datos de delitos por fecha-hora |

**Endpoints importantes:**
- **Histórico (2020–2024):** `https://archive-api.open-meteo.com/v1/archive`
- **Pronóstico (próximos 7 días):** `https://api.open-meteo.com/v1/forecast`

**🔄 Frecuencia de actualización:** Horaria / tiempo real  
**📁 Formato:** JSON via API REST  
**📐 Volumen:** Ilimitado (API paginable por fecha)  
**🏷️ Licencia:** CC BY 4.0  

**⚠️ Advertencia de calidad:**  
Rate limit de 10.000 requests por día para uso sin clave (más que suficiente para el proyecto). El histórico completo 2020–2024 se descarga en una sola llamada.

**💡 Por qué es clave:**  
Es la única variable del modelo que actualiza en tiempo real. Sin datos de clima, el modelo solo puede predecir patrones históricos. Con lluvia y temperatura, puede ajustar la predicción para las próximas horas — esto es lo que hace la diferencia frente a un análisis descriptivo simple.


In [ ]:
# ──────────────────────────────────────────────────────────────
# FUENTE 3 — Descarga de datos climáticos Bogotá (Open-Meteo)
# No requiere clave API ni registro
# ──────────────────────────────────────────────────────────────
import requests
import pandas as pd

BOG_LAT = 4.6097
BOG_LON = -74.0817

def cargar_clima(fecha_inicio='2020-01-01', fecha_fin='2024-12-31'):
    """Descarga clima horario de Bogotá desde Open-Meteo (histórico)."""
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude':   BOG_LAT,
        'longitude':  BOG_LON,
        'start_date': fecha_inicio,
        'end_date':   fecha_fin,
        'hourly':     'temperature_2m,precipitation,windspeed_10m,relativehumidity_2m',
        'timezone':   'America/Bogota'
    }
    resp = requests.get(url, params=params, timeout=60)
    data = resp.json()
    df = pd.DataFrame(data['hourly'])
    df['time'] = pd.to_datetime(df['time'])
    df = df.rename(columns={
        'time':                 'DATETIME',
        'temperature_2m':       'TEMPERATURA_C',
        'precipitation':        'PRECIPITACION_MM',
        'windspeed_10m':        'VIENTO_KMH',
        'relativehumidity_2m':  'HUMEDAD_PCT'
    })
    df['FECHA'] = df['DATETIME'].dt.date
    df['HORA']  = df['DATETIME'].dt.hour
    print(f'✅ Clima cargado: {len(df):,} registros horarios')
    print(f'   Rango: {df["DATETIME"].min()} → {df["DATETIME"].max()}')
    print(f'   Temperatura promedio Bogotá: {df["TEMPERATURA_C"].mean():.1f}°C')
    return df

# df_clima = cargar_clima()
print('▶ Descomenta la línea anterior para descargar el clima. No necesita clave API.')


---
### 🟢 FUENTE 4 — Cuadrantes de Policía Bogotá · MEBOG

**Organización:** MEBOG — Policía Metropolitana de Bogotá  
**Portal:** datosabiertos.bogota.gov.co  
**Plataforma:** CKAN — descarga directa GeoJSON / Shapefile

**🔗 Enlace directo:**
```
https://datosabiertos.bogota.gov.co/dataset/cuadrantes-de-policia-bogota-d-c
```

**📋 Qué encontrarás en ese enlace:**  
Los aproximadamente 1.200 cuadrantes de Policía de Bogotá en formato GeoJSON o Shapefile. Cada cuadrante es el territorio asignado a una patrulla de la Policía. Esta fuente es el puente entre el modelo y la acción: permite conectar el riesgo predicho en cada UPZ con el cuadrante específico y el CAI responsable de actuar.

**📊 Variables que usaremos para el proyecto:**

| Variable | Descripción | Cómo la usamos |
|----------|-------------|----------------|
| `codigo_cuadrante` | Código identificador del cuadrante (ej. CHAP-07) | Citar en la recomendación del Módulo 3 |
| `geometry` | Polígono del cuadrante | Spatial join cuadrante → UPZ |
| `area` | Área en m² | Calcular cobertura policial por UPZ |
| `localidad` | Localidad donde está el cuadrante | Agrupación y filtrado |
| `nombre_cai` | Nombre del CAI al que pertenece | Mostrar al comandante en la recomendación |

**Feature generada:** `cuadrantes_por_km2_upz` — densidad de cobertura policial por UPZ. UPZs con menos cobertura tienen más riesgo estructural.

**🔄 Frecuencia de actualización:** Anual (última actualización: marzo 2026)  
**📁 Formato:** GeoJSON / SHP ZIP  
**📐 Volumen:** ~1.200 cuadrantes  
**🏷️ Licencia:** Pública  

**⚠️ Advertencia de calidad:**  
El dataset contiene solo geometría. No incluye datos de actividad policial (llamadas atendidas, patrullajes). Para eso existe FUENTE 5 (NUSE 123).

**💡 Por qué es clave:**  
Sin esta fuente, el Módulo 3 (Recomendación) no puede decirle al comandante a qué cuadrante específico debe ir. Es la bisagra entre la predicción y la acción institucional — el diferenciador más importante del proyecto frente a CrimeLab Santander 2025.


In [ ]:
# ──────────────────────────────────────────────────────────────
# FUENTE 4 — Cuadrantes de Policía y su relación con UPZ
# ──────────────────────────────────────────────────────────────
import geopandas as gpd

# El archivo se descarga desde el portal (GeoJSON o SHP dentro de ZIP)
# Guárdalo en datos/raw/cuadrantes_policia.geojson

def cargar_cuadrantes(ruta='datos/raw/cuadrantes_policia.geojson'):
    try:
        cuad = gpd.read_file(ruta)
        cuad = cuad.to_crs('EPSG:4326')
        print(f'✅ Cuadrantes cargados: {len(cuad):,}')
        print(f'   Columnas: {cuad.columns.tolist()}')
        return cuad
    except FileNotFoundError:
        print('⚠️  Archivo no encontrado.')
        print('   Descarga el GeoJSON desde el enlace de FUENTE 4.')
        return None

def cuadrantes_por_upz(cuad, upz):
    """Calcula cuántos cuadrantes caben en cada UPZ y genera la feature de densidad."""
    join = gpd.sjoin(upz[['codigo_upz', 'geometry']], cuad, how='left', predicate='intersects')
    conteo = join.groupby('codigo_upz').size().reset_index(name='n_cuadrantes')

    # Área en km²
    upz_m = upz.to_crs('EPSG:3116').copy()
    upz_m['area_km2'] = upz_m.geometry.area / 1e6
    resultado = conteo.merge(upz_m[['codigo_upz', 'area_km2']], on='codigo_upz', how='left')
    resultado['cuadrantes_por_km2'] = resultado['n_cuadrantes'] / resultado['area_km2']

    print(f'✅ Feature cuadrantes_por_km2_upz generada')
    print(f'   Promedio Bogotá: {resultado["cuadrantes_por_km2"].mean():.1f} cuadrantes/km²')
    return resultado

print('▶ Carga cuadrantes y upz primero, luego llama cuadrantes_por_upz(cuad, upz).')


---
### 🟢 FUENTE 5 — Incidentes NUSE 123 · C4 ⭐ FUENTE BASE DEL MODELO

**Organización:** Secretaría Distrital de Seguridad, Convivencia y Justicia  
**Portal:** datosabiertos.bogota.gov.co  
**Plataforma:** CKAN Datastore — API paginada (sin descarga completa de una vez)

**🔗 Enlace de la API:**
```
https://datosabiertos.bogota.gov.co/api/3/action/datastore_search?resource_id=30d65a8b-d0ed-4e95-977e-0d7cc2ea89ef&limit=1000
```

**📋 Qué encontrarás en ese enlace:**  
Las llamadas al número de emergencias 123 de Bogotá, con **86 tipos de incidente** por UPZ × mes × tipo. Cubre enero 2025 – abril 2026 (128,314 registros).

> **⭐ HALLAZGO FASE 1B:** F5 es la **fuente principal** del proyecto — la única con granularidad UPZ × mes × tipo de incidente. Genera las 111,606 filas de la tabla Silver. Las otras fuentes (F3/F4/F7/F8) agregan columnas; F5 genera las filas.

**📊 Variables que usaremos para el proyecto:**

| Variable | Descripción | Cómo la usamos |
|----------|-------------|----------------|
| `COD_UPZ` | Código UPZ (formato `UPZ99`) | Llave de unión — unidad de análisis |
| `ANIO` | Año del incidente | Unión temporal |
| `MES` | Mes del incidente | Unión temporal |
| `TIPO_DETALLE` | Tipo de incidente (86 tipos: HURTO, RIÑA, ACCIDENTE…) | Columna `tipo_crimen` en Silver |
| `CANT_INCIDENTES` | Conteo mensual por UPZ × tipo | `n_delitos` en Silver |
| `COD_LOCALIDAD` / `LOCALIDAD` | Código y nombre de localidad | `cod_localidad`, `nom_localidad` en Silver |

**Features generadas en Silver:**
- `es_crimen` — True para los 19 tipos de alto impacto criminal
- `n_delitos_upz_4sem` — lag 1 mes (≈4 semanas)
- `n_delitos_upz_8sem` — lag acumulado 2 meses
- `ratio_nuse_delitos_upz` — proxy de subregistro

**🔄 Frecuencia de actualización:** Mensual (última: abril 2026)  
**📁 Formato:** API JSON paginada (10K registros/llamada)  
**📐 Volumen Bronze:** 128,314 registros → **111,606 filas Silver** (86 tipos × 120 UPZs × 16 meses)  
**🏷️ Licencia:** CC BY-SA 4.0  

**⚠️ Nota:** Solo disponible 2025–2026. Split temporal del modelo: TRAIN = ene–oct 2025, TEST = nov 2025–abr 2026.

**💡 Por qué es la fuente base:**  
Es la única fuente con desglose UPZ × mes × tipo — las tres dimensiones necesarias para el modelo. Su cobertura de los 86 tipos de incidente incluye tanto crímenes de alto impacto como desorden urbano, permitiendo que el modelo aprenda correlaciones entre ambos.

In [ ]:
# ──────────────────────────────────────────────────────────────
# FUENTE 5 — Descarga de incidentes NUSE 123 via API CKAN
# ──────────────────────────────────────────────────────────────
import requests
import pandas as pd

RESOURCE_ID_NUSE = '30d65a8b-d0ed-4e95-977e-0d7cc2ea89ef'
BASE_URL = 'https://datosabiertos.bogota.gov.co/api/3/action/datastore_search'

def cargar_nuse(resource_id=RESOURCE_ID_NUSE, max_registros=200_000):
    """Descarga todos los incidentes NUSE 123 paginando la API."""
    todos = []
    offset = 0
    batch = 10_000
    while len(todos) < max_registros:
        resp = requests.get(BASE_URL, params={
            'resource_id': resource_id,
            'limit': batch,
            'offset': offset
        }, timeout=30)
        records = resp.json()['result']['records']
        if not records:
            break
        todos.extend(records)
        offset += batch
        print(f'   Descargados {len(todos):,} registros...', end='\r')
    df = pd.DataFrame(todos)
    print(f'\n✅ NUSE 123 cargado: {len(df):,} registros')
    print(f'   Columnas: {df.columns.tolist()}')
    return df

# df_nuse = cargar_nuse()
print('▶ Descomenta la línea anterior para descargar los datos NUSE.')
print('   Puede tardar 2-3 minutos por el volumen de datos.')


---
### 🟢 FUENTE 6 — Hurto a Personas · Policía Nacional

**Organización:** Policía Nacional de Colombia  
**Portal:** datos.gov.co  
**Plataforma:** Socrata — API REST (sodapy)

**🔗 Enlace de la API:**
```
https://www.datos.gov.co/resource/4rxi-8m8d.json
```

**📋 Qué encontrarás en ese enlace:**  
Registro de todos los hurtos a personas en Colombia, actualizados mensualmente por la Policía Nacional. Datos disponibles desde 2010. Permite comparar las cifras de Bogotá frente al promedio nacional, analizar tendencias de largo plazo y validar cruzadamente los datos del Delito de Alto Impacto (FUENTE 1). Es la fuente nacional de referencia.

**📊 Variables que usaremos para el proyecto:**

| Variable | Descripción | Cómo la usamos |
|----------|-------------|----------------|
| `fecha_hecho` | Fecha del evento | Tendencia mensual / anual |
| `departamento` | Departamento | Filtrar solo Cundinamarca / Bogotá |
| `municipio` | Ciudad | Filtrar: `BOGOTA D.C.` |
| `zona` | Urbano / rural | Contexto — filtrar urbano |
| `cantidad` | Número de hurtos en esa categoría | Tendencia comparativa |

**🔄 Frecuencia de actualización:** Mensual (datos de abril 2026 disponibles)  
**📁 Formato:** API JSON (Socrata) — usar librería `sodapy` o `requests`  
**📐 Volumen:** +500.000 registros (desde 2010)  
**🏷️ Licencia:** Datos abiertos públicos  

**⚠️ Advertencia de calidad:**  
Sin App Token de Socrata: límite de 1.000 filas por request. Hay que paginar con `$offset`. Solo llega al nivel municipal, no a UPZ. Se usa para contexto y benchmarking, no como feature directa del modelo.

**💡 Por qué la incluimos:**  
Permite al jurado ver que el proyecto no solo usa datos de Bogotá sino que los contextualiza frente al nivel nacional. También sirve para validar que los datos de FUENTE 1 son consistentes con las estadísticas oficiales de la Policía.


In [ ]:
# ──────────────────────────────────────────────────────────────
# FUENTE 6 — Hurto a Personas Policía Nacional (API Socrata)
# ──────────────────────────────────────────────────────────────
import requests
import pandas as pd

def cargar_socrata(resource_id, filtro_municipio='BOGOTA D.C.', max_registros=50_000):
    """Descarga datos de la API Socrata de datos.gov.co con paginación."""
    url = f'https://www.datos.gov.co/resource/{resource_id}.json'
    todos = []
    offset = 0
    batch = 1000  # Límite sin App Token
    while len(todos) < max_registros:
        params = {'$limit': batch, '$offset': offset}
        if filtro_municipio:
            params['municipio'] = filtro_municipio
        resp = requests.get(url, params=params, timeout=30)
        data = resp.json()
        if not data:
            break
        todos.extend(data)
        offset += batch
        if len(data) < batch:
            break
        print(f'   {len(todos):,} registros...', end='\r')
    return pd.DataFrame(todos)

# Hurto a personas — Bogotá
# df_hurto = cargar_socrata('4rxi-8m8d')
# print(f'\n✅ Hurto Bogotá: {len(df_hurto):,} registros')
print('▶ Descomenta para descargar hurto a personas (Policía Nacional).')


---
### 🟡 FUENTE 7 — Estratificación por Manzana · SDP

**Organización:** Secretaría Distrital de Planeación de Bogotá  
**Portal:** datosabiertos.bogota.gov.co  
**Plataforma:** CKAN — descarga directa GeoJSON

**🔗 Enlace de descarga:**
```
https://datosabiertos.bogota.gov.co/dataset/55467552-0af4-4524-a390-a2956035744e/resource/29f2d770-bd5d-4450-9e95-8737167ba12f/download/manzanaestratificacion.json
```

**📋 Qué encontrarás en ese enlace:**  
El estrato socioeconómico (1 a 6) de cada manzana catastral de Bogotá, con más de 100.000 manzanas. Última actualización: noviembre 2025. El estrato es una de las variables más correlacionadas con la criminalidad en ciudades colombianas. Además cumple un rol analítico fundamental: permite responder si el modelo predice más riesgo en zonas de estrato bajo simplemente porque hay más datos ahí, o porque el riesgo es genuinamente mayor.

**📊 Variables que usaremos para el proyecto:**

| Variable | Descripción | Cómo la usamos |
|----------|-------------|----------------|
| `estrato` | Estrato socioeconómico (1–6) de la manzana | Promedio por UPZ → feature del modelo |
| `geometry` | Polígono de la manzana catastral | Spatial join con UPZ para agregar |
| `localidad` | Localidad de la manzana | Contexto geográfico |

**Feature generada:** `estrato_promedio_upz` — promedio ponderado por área del estrato de todas las manzanas dentro de cada UPZ.

**🔄 Frecuencia de actualización:** Según necesidad (última: noviembre 2025)  
**📁 Formato:** GeoJSON  
**📐 Volumen:** +100.000 manzanas  
**🏷️ Licencia:** CC BY 4.0  

**⚠️ Advertencia de calidad:**  
El spatial join manzana → UPZ es computacionalmente costoso (más de 100.000 polígonos). Se recomienda pre-calcular el promedio por UPZ y guardarlo como CSV o Parquet antes de correr el modelo.

**💡 Por qué la incluimos:**  
El jurado del concurso casi siempre pregunta: *"¿El modelo no estará simplemente discriminando por estrato?"*. Con esta fuente se puede responder con evidencia: mostrar el análisis de equidad y demostrar que el modelo predice basado en patrones de criminalidad, no solo en condición socioeconómica.


In [ ]:
# ──────────────────────────────────────────────────────────────
# FUENTE 7 — Estratificación por manzana → Feature por UPZ
# ──────────────────────────────────────────────────────────────
import geopandas as gpd
import pandas as pd

URL_ESTRATIFICACION = (
    'https://datosabiertos.bogota.gov.co/dataset/'
    '55467552-0af4-4524-a390-a2956035744e/resource/'
    '29f2d770-bd5d-4450-9e95-8737167ba12f/download/manzanaestratificacion.json'
)

def calcular_estrato_por_upz(upz, url_estratificacion=URL_ESTRATIFICACION):
    """
    Calcula el estrato promedio por UPZ haciendo spatial join
    manzana catastral → UPZ.
    NOTA: Puede tardar varios minutos por el volumen de manzanas.
    """
    print('🔄 Cargando manzanas (puede tardar 2-5 minutos)...')
    manzanas = gpd.read_file(url_estratificacion)
    manzanas = manzanas.to_crs('EPSG:4326')

    # Tomar el centroide de cada manzana para acelerar el join
    manzanas_c = manzanas.copy()
    manzanas_c['geometry'] = manzanas_c.geometry.centroid

    print('🔄 Haciendo spatial join manzana → UPZ...')
    join = gpd.sjoin(upz[['codigo_upz', 'geometry']], manzanas_c[['estrato', 'geometry']],
                     how='left', predicate='contains')

    estrato_upz = (
        join.groupby('codigo_upz')['estrato']
        .agg(lambda x: pd.to_numeric(x, errors='coerce').mean())
        .reset_index()
        .rename(columns={'estrato': 'estrato_promedio_upz'})
    )
    print(f'✅ Estrato promedio calculado para {len(estrato_upz)} UPZs')
    # Guardar para no recalcular cada vez
    estrato_upz.to_csv('datos/procesados/estrato_por_upz.csv', index=False)
    print('   Guardado en datos/procesados/estrato_por_upz.csv')
    return estrato_upz

# estrato_upz = calcular_estrato_por_upz(upz)
print('▶ Descomenta cuando tengas upz cargada. Guarda el resultado — es costoso recalcular.')


---
### 🟢 FUENTE 8 — Estaciones de TransMilenio · TransMilenio S.A.

**Organización:** TransMilenio S.A.  
**Portal:** datosabiertos.bogota.gov.co  
**Plataforma:** CKAN — descarga directa / ArcGIS REST

**🔗 Enlace directo:**
```
https://datosabiertos.bogota.gov.co/dataset/9be8b6fb-8059-492f-a866-4a1ac031c502
```

**📋 Qué encontrarás en ese enlace:**  
La ubicación geoespacial de todas las estaciones troncales, portales y cables del sistema TransMilenio y SITP de Bogotá. Los nodos de alta afluencia de pasajeros (portales, estaciones de integración, estaciones de las troncales principales) concentran hurto a personas por la sencilla razón de que hay más personas: más víctimas potenciales, más anonimato, más oportunidad para el delincuente. La distancia de una UPZ a la estación TM más cercana captura el nivel de exposición a flujos masivos de personas.

**📊 Variables que usaremos para el proyecto:**

| Variable | Descripción | Cómo la usamos |
|----------|-------------|----------------|
| `nombre_estacion` | Nombre de la estación | Etiqueta en el mapa y chatbot |
| `lat` / `lon` | Coordenadas GPS | Spatial join con UPZ |
| `tipo` | Troncal / cable / portal / intermedia | Peso diferencial (portales = más afluencia) |
| `corredor` | Línea TM (NQS, Caracas, Suba…) | Análisis por corredor |

**Features generadas:**
- `n_estaciones_tm_upz` — número de estaciones dentro de la UPZ
- `distancia_tm_mas_cercana` — distancia en metros a la estación TM más cercana (para UPZs sin estación propia)

**🔄 Frecuencia de actualización:** Según necesidad (última actualización: julio 2025)  
**📁 Formato:** GeoJSON / Shapefile  
**📐 Volumen:** ~150 estaciones troncales  
**🏷️ Licencia:** Pública  

**⚠️ Advertencia de calidad:**  
La red TM crece lentamente (1–2 estaciones por año), así que el dato de 2025 es suficientemente actual. No incluye datos de afluencia por estación (esos son de TransMilenio directamente y no están publicados como datos abiertos).

**💡 Por qué la incluimos:**  
Los patrones de hurto en Bogotá tienen una correlación espacial muy clara con las estaciones de TransMilenio. Incluir esta variable en el modelo lo hace más preciso y al mismo tiempo más explicable: *"El riesgo alto en la UPZ Chapinero se debe en parte a que concentra 4 estaciones de TransMilenio en su territorio"*.


In [ ]:
# ──────────────────────────────────────────────────────────────
# FUENTE 8 — Estaciones TransMilenio → Features por UPZ
# ──────────────────────────────────────────────────────────────
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

def cargar_transmilenio(ruta='datos/raw/estaciones_transmilenio.geojson'):
    try:
        tm = gpd.read_file(ruta)
        tm = tm.to_crs('EPSG:4326')
        print(f'✅ Estaciones TM cargadas: {len(tm)}')
        print(f'   Tipos: {tm["tipo"].value_counts().to_dict() if "tipo" in tm.columns else "ver columnas"}')
        return tm
    except FileNotFoundError:
        print('⚠️  Descarga el archivo GeoJSON desde el enlace de FUENTE 8')
        print('   y guárdalo en datos/raw/estaciones_transmilenio.geojson')
        return None

def features_transmilenio_por_upz(upz, tm):
    """Genera n_estaciones_tm_upz y distancia_tm_mas_cercana."""
    # Estaciones dentro de cada UPZ
    join = gpd.sjoin(upz[['codigo_upz', 'geometry']], tm, how='left', predicate='contains')
    n_estaciones = join.groupby('codigo_upz').size().reset_index(name='n_estaciones_tm_upz')

    # Distancia a la estación más cercana (en metros, CRS métrico)
    upz_m  = upz.to_crs('EPSG:3116').copy()
    tm_m   = tm.to_crs('EPSG:3116').copy()
    centroides = upz_m.copy()
    centroides['geometry'] = centroides.geometry.centroid
    distancias = []
    for _, row in centroides.iterrows():
        dist_min = tm_m.geometry.distance(row.geometry).min()
        distancias.append({'codigo_upz': row['codigo_upz'], 'dist_tm_metros': round(dist_min)})
    df_dist = pd.DataFrame(distancias)

    resultado = n_estaciones.merge(df_dist, on='codigo_upz', how='outer').fillna(0)
    print(f'✅ Features TM generadas para {len(resultado)} UPZs')
    return resultado

# tm = cargar_transmilenio()
# if tm is not None and 'upz' in dir():
#     features_tm = features_transmilenio_por_upz(upz, tm)
print('▶ Descomenta cuando tengas upz y tm cargadas.')


---
### ✅ Resumen de las 10 Fuentes Activas + 2 Planificadas

| # | Fuente | Organización | Filas Bronze | Rol en el pipeline | Semáforo |
|---|--------|-------------|:---:|------------------|----------|
| F1 | Delito de Alto Impacto — Bogotá | Sec. Seguridad Bogotá | 21 | EDA histórico 2018–2026 (no tiene UPZ) | 🟢 |
| F2 | UPZ Shapefile — IDECA | IDECA | 112 | **Capa base espacial — todos los spatial joins** | 🟢 |
| F3 | Open-Meteo — Clima Bogotá | Open-Meteo | ~56K | Features climáticas (temperatura, lluvia) | 🟢 |
| F4 | Cuadrantes Policía — MEBOG | Policía MEBOG | 599 | Feature cobertura policial + nombre CAI | 🟢 |
| **F5** | **Incidentes NUSE 123 — C4** ⭐ | **Sec. Seguridad Bogotá** | **128,314** | **BASE del modelo → 111,606 filas Silver** | 🟢 |
| F6 | Hurto Personas — Policía Nacional | Policía Nacional | 638,569 | Benchmarking nacional (no tiene UPZ) | 🟢 |
| F7 | Estratificación manzanas — SDP | Sec. Planeación | 44,260 | Feature socioeconómica + análisis equidad | 🟡 |
| F8 | Estaciones TransMilenio | TransMilenio S.A. | 153 | Features movilidad / afluencia | 🟢 |
| F9 | Boletines SCJ — Sec. Seguridad | scj.gov.co | N/A (PDFs) | Corpus LLM → GraphRAG + Claude API | 🟢 |
| F10 | Noticias RSS — El Tiempo / Espectador | Medios públicos | N/A (RSS) | Corpus LLM → GraphRAG + Claude API | 🟢 |
| F11 *(planificada)* | IDU Calzada — Estado Superficial | IDU | ~miles | Feature `km_via_intervenida_upz` → XGBoost | ⏳ Fase 2 |
| F12 *(planificada)* | Plan Desarrollo Bogotá 2024-2027 | SDP — Acuerdo 927 | 1 PDF | Corpus LLM → GraphRAG | ⏳ Fase 3 |

**Silver final:** 111,606 filas × 20 columnas — llave: `upz_cod × anio × mes × tipo_crimen`  
**Fuentes en datos.gov.co / datosabiertos.bogota.gov.co:** F1, F4, F5, F6, F7 ✅  
**F9/F10** no entran en XGBoost — son corpus de texto para Claude API (Módulos 3 y 4)

---
## 🏗️ SECCIÓN 5 — Arquitectura Técnica

### Pipeline de datos — Arquitectura Medallón

Adoptamos la misma arquitectura Bronze / Silver / Gold que usó CrimeLab Santander 2025 (funciona muy bien para proyectos de este tipo), y le añadimos la capa de IA Generativa.

```
datos/
├── raw/           ← Bronze: archivos tal como se descargan (ZIP, GeoJSON, CSV)
├── procesados/    ← Silver: datos limpios, unificados, coordenadas validadas
├── features/      ← Gold:   tabla maestra por UPZ con las 14 variables del modelo
└── modelos/       ← Model:  modelo XGBoost entrenado + valores SHAP
```

### Stack tecnológico

| Capa | Tecnologías |
|------|-------------|
| **Ingesta y limpieza** | `pandas`, `geopandas`, `requests` |
| **Análisis espacial** | `geopandas`, `folium`, `shapely` |
| **Modelo ML** | `xgboost`, `scikit-learn`, `shap` |
| **IA Generativa** | `Claude API` (Anthropic) |
| **Dashboard** | `streamlit`, `plotly`, `folium` |
| **Repositorio** | `GitHub` (público, obligatorio para el concurso) |

### Variables del modelo XGBoost

```
FEATURES DE ENTRADA (X):
  Históricas:   n_delitos_upz_4sem, n_delitos_upz_8sem, tipo_delito_dominante
  Temporales:   dia_semana, franja_horaria, mes, es_fin_semana
  Climáticas:   temperatura_c, precipitacion_mm  ← Open-Meteo
  Espaciales:   estrato_promedio_upz, cuadrantes_por_km2, n_estaciones_tm, dist_tm_metros
  Subregistro:  ratio_nuse_delitos_upz

VARIABLE OBJETIVO (Y):
  nivel_riesgo: ALTO / MEDIO / BAJO
```


---
## 📅 SECCIÓN 6 — Cronograma de Fases

| Fase | Fechas | Qué se construye | Entregable |
|------|--------|------------------|------------|
| **✅ Fase 0** | 23 May | Propuesta + catálogo de 12 fuentes (F1-F10 + F11/F12 planificadas) | Este notebook |
| **✅ Fase 1** | 26 May – 6 Jun | Descarga y EDA de las 10 fuentes (F1-F10) · Silver 111,606 filas × 20 cols | Notebook EDA + `silver_upz_mes.parquet` |
| **⏳ Fase 2** | 7 – 20 Jun | Modelo XGBoost + SHAP + F11 IDU | Notebooks 03+04 · modelo entrenado · mapa predictivo |
| **⏳ Fase 3** | 21 Jun – 10 Jul | Dashboard Streamlit + GraphRAG + Claude API + F12 | App completa 4 módulos en Streamlit Cloud |
| **⏳ Fase 4** | 11 – 13 Jul | Documentación + video + publicación | README, CRISP-ML, video pitch 3 min, registro datos.gov.co |

### Hito crítico: **13 de julio de 2026** *(verificar posible extensión a agosto — GovCamps 2026)*
Publicar en GitHub + registrar en la sección "Usos" de datos.gov.co. **Obligatorio** para avanzar a la sustentación virtual (14–17 julio).

---
## 🏆 SECCIÓN 7 — Criterios de Evaluación del Concurso

El jurado evalúa 6 criterios sobre 100 puntos. Aquí está cómo SeguroData responde a cada uno:

| Criterio | Pts | Cómo lo cubrimos | Puntaje estimado |
|----------|-----|------------------|-----------------|
| **Uso de datos abiertos** | 20 | 10 fuentes activas (12 con planificadas) — múltiples portales datos.gov.co / datosabiertos.bogota.gov.co | **18 / 20** |
| **IA y tecnologías emergentes** | 20 | XGBoost + SHAP + Claude API (GraphRAG) + clima en tiempo real | **18 / 20** |
| **Impacto y escalabilidad** | 20 | Conexión directa con CAI de Bogotá · 3 usuarios · escalable a Medellín y Cali | **17 / 20** |
| **Innovación** | 15 | GraphRAG causal (explica el *por qué*) + capa prescriptiva con entidades responsables | **13 / 15** |
| **Rigor técnico** | 15 | Backtesting temporal (no aleatorio) · SHAP · CRISP-ML documentado · análisis de sesgo | **12 / 15** |
| **Diseño y usabilidad** | 10 | Dashboard Streamlit limpio · chatbot ciudadano · 3 vistas de usuario | **9 / 10** |
| **TOTAL ESTIMADO** | **100** | | **87 / 100** |

### El argumento central ante el jurado

*"SeguroData no solo predice dónde habrá delitos — le dice exactamente a qué cuadrante de la Policía tiene que ir y **por qué**: qué obra, qué política, qué patrón histórico causó ese riesgo. Ese es el puente que hoy no existe entre los datos abiertos de Bogotá y la acción institucional."*

---
## ⚙️ SECCIÓN 8 — Configuración del Entorno

Ejecuta estas celdas antes de cualquier otra cosa.


In [ ]:
# ──────────────────────────────────────────────────────────────
# INSTALACIÓN DE DEPENDENCIAS
# Ejecutar solo la primera vez
# ──────────────────────────────────────────────────────────────
!pip install pandas geopandas folium plotly requests xgboost              scikit-learn shap streamlit openpyxl shapely -q
print('✅ Dependencias instaladas')


In [ ]:
# ──────────────────────────────────────────────────────────────
# ESTRUCTURA DE CARPETAS
# ──────────────────────────────────────────────────────────────
import os, warnings
warnings.filterwarnings('ignore')

carpetas = [
    'datos/raw',
    'datos/procesados',
    'datos/features',
    'datos/modelos',
    'graficas',
]
for carpeta in carpetas:
    os.makedirs(carpeta, exist_ok=True)

print('✅ Estructura de carpetas lista:')
for c in carpetas:
    print(f'   📁 {c}/')


In [ ]:
# ──────────────────────────────────────────────────────────────
# CONSTANTES GLOBALES DEL PROYECTO
# ──────────────────────────────────────────────────────────────
import pandas as pd

# Coordenadas de Bogotá
BOG_LAT = 4.6097
BOG_LON = -74.0817

# Límites válidos de coordenadas para Bogotá
BOG_LAT_MIN, BOG_LAT_MAX = 3.7, 4.9
BOG_LON_MIN, BOG_LON_MAX = -74.4, -73.9

# Años de análisis
AÑOS_ANALISIS = list(range(2020, 2025))

# Nombres de días y meses en español
NOMBRES_DIAS = {
    0:'Lunes', 1:'Martes', 2:'Miércoles',
    3:'Jueves', 4:'Viernes', 5:'Sábado', 6:'Domingo'
}
NOMBRES_MESES = {
    1:'Ene', 2:'Feb', 3:'Mar', 4:'Abr', 5:'May', 6:'Jun',
    7:'Jul', 8:'Ago', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dic'
}

print('✅ Constantes globales definidas')
print(f'   Ciudad: Bogotá D.C. ({BOG_LAT}°N, {BOG_LON}°W)')
print(f'   Años de análisis: {AÑOS_ANALISIS}')
print()
print('📋 CHECKLIST DE DESCARGA MANUAL:')
print('   [ ] FUENTE 1 → dai_geojson.zip → datos/raw/')
print('   [ ] FUENTE 2 → se carga directo desde URL')
print('   [ ] FUENTE 3 → se carga directo desde API')
print('   [ ] FUENTE 4 → cuadrantes_policia.geojson → datos/raw/')
print('   [ ] FUENTE 5 → se descarga via API')
print('   [ ] FUENTE 6 → se descarga via API')
print('   [ ] FUENTE 7 → se carga directo desde URL')
print('   [ ] FUENTE 8 → estaciones_transmilenio.geojson → datos/raw/')


---
## 🚀 Próximo paso: Notebook 02 — EDA ✅ Completado

Las 10 fuentes han sido descargadas y transformadas en el pipeline Silver. El Notebook 02 realiza el análisis exploratorio completo sobre la tabla `silver_upz_mes.parquet` (111,606 filas × 20 columnas):

- Distribución de incidentes por UPZ y localidad (V1 choropleta, V3 top-10)
- Heatmap tipo de incidente × mes (V2)
- Tendencia histórica 2018–2026 usando F1 DAI + F6 Hurto PN (V4)
- Correlación lluvia/temperatura vs. incidentes (V5)
- Distribución por estrato socioeconómico — base del análisis de sesgo (V6)
- Distribución de `n_delitos` y cobertura policial (V7)

**Siguiente fase:** Notebook 03 — Feature Engineering → tabla maestra Gold con las 14 variables del modelo.

---
*SeguroData Bogotá · Concurso Datos al Ecosistema 2026 · MinTIC · Reto #2 · Nivel Medio*